In [ ]:
# nanoGPT from scratch on Kaggle
# 第 1 个 cell：环境检查、目标说明、导入依赖、固定随机种子
#
# 这份 notebook 的目标不是直接调用官方 nanoGPT 的 train.py，
# 而是在 Kaggle 里一步一步“自己写出一个 nanoGPT”。
# 官方源码已经放在仓库的 nanoGPT-reference/ 目录中，后续每一步都会对照它来实现。
#
# 当前第 1 格只做准备工作：
# 1. 确认 PyTorch 和 GPU 状态
# 2. 处理 Kaggle 上 P100 + 新版 PyTorch 不兼容的问题
# 3. 导入后续会用到的基础库
# 4. 固定随机种子，方便复现实验
# 5. 设置一个 device 变量，后续张量和模型都放到这个设备上

import math
import os
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F


print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


def choose_device():
    """选择当前 Kaggle 环境里真正能安全使用的设备。"""
    if not torch.cuda.is_available():
        print("GPU not found, using CPU. 也能跑小实验，但会慢一些。")
        return "cpu"

    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    print("GPU:", name)
    print("CUDA capability:", f"sm_{props.major}{props.minor}")

    # Kaggle 有时会分配 Tesla P100。P100 是 sm_60。
    # 你遇到的 PyTorch 2.10.0+cu128 构建只支持 sm_70 及以上。
    # 这种情况下 torch.cuda.is_available() 仍然可能是 True，
    # 但真正执行 CUDA kernel 时会失败，所以这里主动退回 CPU。
    if props.major < 7:
        print("\n注意：当前 GPU 是 P100/sm_60，但这个 PyTorch 构建不支持 sm_60。")
        print("本 notebook 将临时使用 CPU。想用 GPU，请在 Kaggle Accelerator 里优先选择 T4。")
        print("如果 Kaggle 只能给 P100，也可以改装兼容旧架构的 PyTorch，但初学阶段先不折腾环境。\n")
        return "cpu"

    return "cuda"


device = choose_device()


# 固定随机种子。
# 训练神经网络时，初始化权重、打乱数据、采样 token 都会用到随机数。
# 固定 seed 后，每次从头运行 notebook，结果更容易复现和对比。
seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if device == "cuda":
    torch.cuda.manual_seed(seed)


# 允许 TensorFloat-32。它是 NVIDIA GPU 上的一种加速格式。
# 对我们这种学习实验来说，可以让矩阵乘法更快。
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


print("Using device:", device)
print("Seed:", seed)


# 后续我们会按这个顺序继续写：
# Cell 2：准备一个极小文本数据集，并把字符变成整数 token
# Cell 3：写 get_batch，构造 x/y 训练样本
# Cell 4：先实现 BigramLanguageModel，理解最小语言模型
# Cell 5：实现单头 causal self-attention
# Cell 6：实现多头注意力
# Cell 7：实现 MLP 和 Block
# Cell 8：组装 GPTConfig 和 GPT
# Cell 9：写训练循环
# Cell 10：写 generate 生成文本
